<a href="https://colab.research.google.com/github/PLucenaa/PLucenaa/blob/main/VERS%C3%83O_OFICIAL_Intera%C3%A7%C3%A3o_de_Medicamentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1 - Instalando Bibliotecas**

In [ ]:
!pip install -q -U google-generativeai

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.4/320.4 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [ ]:
import google.generativeai as genai
import gradio as gr
import textwrap
import numpy as np
import pandas as pd
import re
import unicodedata
import nltk

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-1.0-pro-latest
models/gemini-1.0-pro
models/gemini-pro
models/gemini-1.0-pro-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-pro-exp-0801
models/gemini-1.5-pro-exp-0827
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-exp-0827
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-1.5-flash-8b-exp-0827
models/gemini-1.5-flash-8b-exp-0924
models/gemini-2.0-flash-exp
models/gemini-exp-1206
models/gemini-exp-1121
models/gemini-exp-1114
models/learnlm-1.5-pro-experimental


In [ ]:
model = genai.GenerativeModel('gemini-1.5-flash-latest')

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# **2 - Montando o Drive no Google Colab e Carregando o DataSet**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_excel('/content/drive/MyDrive/IA na Saúde/Hands-On/Base de Dados/interacao_300.xlsx')
df.head()

,Droga A,Droga B,Mecanismos de Interação
0,Abacavir,Acetato de sódio,O abacavir pode diminuir a taxa de excreção de...
1,Acetazolamida,Acetato de sódio,A acetazolamida pode aumentar a taxa de excreç...
2,Aciclovir,Acetato de sódio,O aciclovir pode diminuir a taxa de excreção d...
3,Ácido acetilsalicílico,Acetato de sódio,O ácido acetilsalicílico pode diminuir a taxa ...
4,Alopurinol,Acetato de sódio,O alopurinol pode diminuir a taxa de excreção ...


# **3 - Tratamento dos Dados**

In [ ]:
def tratar_texto(texto):
    texto = texto.lower()

    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join([c for c in texto if unicodedata.category(c) != 'Mn'])

    texto = re.sub(r'[^\w\s]', ' ', texto)
    texto = re.sub(r'\d+', 'NUM', texto)
    tokens = word_tokenize(texto, language='portuguese')

    stop_words = set(stopwords.words('portuguese'))
    tokens = [word for word in tokens if word not in stop_words]

    stemmer = SnowballStemmer("portuguese")
    tokens = [stemmer.stem(word) for word in tokens]

    texto_preprocessado = ' '.join(tokens)
    return texto_preprocessado

In [ ]:
df['Mecanismos de Interação Tratados'] = df['Mecanismos de Interação'].apply(tratar_texto)
df.head()

,Droga A,Droga B,Mecanismos de Interação,Mecanismos de Interação Tratados
0,Abacavir,Acetato de sódio,O abacavir pode diminuir a taxa de excreção de...,abacav pod diminu tax excreca acetat sodi pod ...
1,Acetazolamida,Acetato de sódio,A acetazolamida pode aumentar a taxa de excreç...,acetazolam pod aument tax excreca acetat sodi ...
2,Aciclovir,Acetato de sódio,O aciclovir pode diminuir a taxa de excreção d...,aciclov pod diminu tax excreca acetat sodi pod...
3,Ácido acetilsalicílico,Acetato de sódio,O ácido acetilsalicílico pode diminuir a taxa ...,acid acetilsalicil pod diminu tax excreca acet...
4,Alopurinol,Acetato de sódio,O alopurinol pode diminuir a taxa de excreção ...,alopurinol pod diminu tax excreca acetat sodi ...


# **4 - Análise de Gravidade**

In [ ]:
def classificar_gravidade(texto):
    texto = texto.lower()
    if re.search(r'(aumenta|aumentar as atividades|risco ou gravidade|perda de eficácia)', texto):
        return "Grave"
    elif re.search(r'(diminui|diminuir|reduz|minimiza)', texto):
        return "Leve"
    elif re.search(r'(pode aumentar|pode diminuir|pode ser aumentado|pode ser diminuído|pode ser aumentada|pode ser diminuída)', texto):
        return "Moderada"

In [ ]:
df['Classificação de Gravidade'] = df['Mecanismos de Interação'].apply(classificar_gravidade)
df.head()

,Droga A,Droga B,Mecanismos de Interação,Mecanismos de Interação Tratados,Classificação de Gravidade
0,Abacavir,Acetato de sódio,O abacavir pode diminuir a taxa de excreção de...,abacav pod diminu tax excreca acetat sodi pod ...,Leve
1,Acetazolamida,Acetato de sódio,A acetazolamida pode aumentar a taxa de excreç...,acetazolam pod aument tax excreca acetat sodi ...,Grave
2,Aciclovir,Acetato de sódio,O aciclovir pode diminuir a taxa de excreção d...,aciclov pod diminu tax excreca acetat sodi pod...,Leve
3,Ácido acetilsalicílico,Acetato de sódio,O ácido acetilsalicílico pode diminuir a taxa ...,acid acetilsalicil pod diminu tax excreca acet...,Leve
4,Alopurinol,Acetato de sódio,O alopurinol pode diminuir a taxa de excreção ...,alopurinol pod diminu tax excreca acetat sodi ...,Leve


In [ ]:
df['Classificação de Gravidade'].value_counts()

,count
Classificação de Gravidade,
Grave,191
Leve,86
Moderada,52


# **5 - Gerando Embeddings**

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['Mecanismos de Interação Tratados'])
df['Embeddings'] = tfidf_matrix.toarray().tolist()
df.head()

,Droga A,Droga B,Mecanismos de Interação,Mecanismos de Interação Tratados,Classificação de Gravidade,Embeddings
0,Abacavir,Acetato de sódio,O abacavir pode diminuir a taxa de excreção de...,abacav pod diminu tax excreca acetat sodi pod ...,Leve,"[0.43778916206971336, 0.0, 0.0, 0.0, 0.3931278..."
1,Acetazolamida,Acetato de sódio,A acetazolamida pode aumentar a taxa de excreç...,acetazolam pod aument tax excreca acetat sodi ...,Grave,"[0.0, 0.0, 0.0, 0.0, 0.3309887007320972, 0.296..."
2,Aciclovir,Acetato de sódio,O aciclovir pode diminuir a taxa de excreção d...,aciclov pod diminu tax excreca acetat sodi pod...,Leve,"[0.0, 0.0, 0.0, 0.0, 0.40063184337280633, 0.0,..."
3,Ácido acetilsalicílico,Acetato de sódio,O ácido acetilsalicílico pode diminuir a taxa ...,acid acetilsalicil pod diminu tax excreca acet...,Leve,"[0.0, 0.0, 0.0, 0.0, 0.38982824212512834, 0.0,..."
4,Alopurinol,Acetato de sódio,O alopurinol pode diminuir a taxa de excreção ...,alopurinol pod diminu tax excreca acetat sodi ...,Leve,"[0.0, 0.0, 0.0, 0.0, 0.3869294626141523, 0.0, ..."


# **6 - Consultando Interações Medicamentosas**

In [ ]:
def buscar_interacao(pergunta):
    pergunta = tratar_texto(pergunta)
    embedding_pergunta = vectorizer.transform([pergunta]).toarray()
    similaridades = cosine_similarity(embedding_pergunta, np.vstack(df['Embeddings']))
    idx_max_similaridade = np.argmax(similaridades)
    resultado = df.iloc[idx_max_similaridade]
    gravidade = resultado['Classificação de Gravidade']
    mecanismo_interacao = resultado['Mecanismos de Interação']

    prompt = (
            f"Considere o seguinte mecanismo de interação medicamentosa: '{mecanismo_interacao}' e a gravidade classificada como '{gravidade}'. "
            f"Crie uma recomendação clara, pequena e breve para o médico sobre o que fazer."
            f"No final do texto, sugira dois medicamentos alternativos que são mais prescritos no Brasil, que possam ser utilizados com segurança e diga para o que eles são indicados.")
    resposta = model.generate_content(prompt)

    if resposta:
        return f"Interação\n{resultado['Mecanismos de Interação']}\n\n"\
               f"Gravidade da Interação\n{resultado['Classificação de Gravidade']}.\n\n"\
               f"Recomendações\n{resposta.text}"
    else:
        return "Nenhuma interação medicamentosa entre os medicamentos. No entanto, não significa que não haja interação."

In [ ]:
def formatar_texto(texto):
    texto = re.sub(r'[^\w\s.,]', '', texto)
    texto = re.sub(r'^\*', '', texto)
    texto = re.sub(r'\. +', '.\n', texto)
    return texto

In [ ]:
pergunta = "Posso prescrever dipirona com paracetamol?"
resultado = buscar_interacao(pergunta)
resultado = formatar_texto(resultado)
print(resultado)

Interação
A dipirona pode diminuir a taxa de excreção de paracetamol, o que pode resultar em um nível sérico mais alto.

Gravidade da Interação
Leve.

Recomendações
Monitorar cuidadosamente os níveis séricos de paracetamol em pacientes que utilizam concomitantemente dipirona.
Considerar redução da dose de paracetamol ou uso alternativo caso necessário.

Alternativas  ibuprofeno antiinflamatório, analgésico, antipirético e naproxeno antiinflamatório, analgésico.



# **7 - Interface com Gradio**

In [ ]:
def gradio_buscar_interacao(pergunta):
    try:
        resposta = buscar_interacao(pergunta)
        resposta = formatar_texto(resposta)
        return resposta
    except Exception as e:
        return f"Erro ao buscar interação: {str(e)}"

iface = gr.Interface(
    fn=buscar_interacao,
    inputs=gr.Textbox(label="Pergunta sobre Interação Medicamentosa", placeholder="Digite sua pergunta aqui...", lines=3),
    outputs=gr.Textbox(label="Resposta", lines=3),
    title="VerificaMed",
    description="Digite uma pergunta sobre interação medicamentosa para obter informações.")
iface.launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://16b72e38ba3068e22e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
